# Signaling Exceptions to the generator

In [78]:
class RetrySignal(Exception):
    pass

def count_from(n: int):
    while True:
        try:
            yield n
            n += 1
        except RetrySignal as e:
            pass

In [79]:
gen = count_from(10)

In [80]:
next(gen)

10

In [81]:
next(gen)

11

In [82]:
next(gen)

12

In [83]:
gen.throw(RetrySignal)

12

In [84]:
gen.throw(RetrySignal)

12

In [85]:
next(gen)

13

In [86]:
next(gen)

14

Notes:
1. `gen.throw()` throws the relevant exception at hthe `yield` inside the generator
2. The generator may handle the exception and do what it needs to do
3. Notice that `gen.throw()` also yields back a value

## A realistic example

### LLM simulator that radomly returns `HTTP 429 - Rate Limit Error`

In [87]:
import random

def call_llm(prompt: str):
    http_status = random.choice([200, 429])
    if http_status != 200:
        print(f'LLM call failed on prompt: {prompt.strip()} - Please retry')
    return http_status

In [88]:
call_llm('hi, how are you ?')

200

### Yield Prompts lazily from a file

Note:
1. Handle `RetrySignal` and yield back same prompt
2. Advance to next prompt if and only if there was no `RetrySignal`

In [70]:
def load_prompts():
    with open('data/prompts.txt', 'r') as f_in:
        for prompt in f_in:
            while True:
                try:
                    yield prompt
                    break
                except RetrySignal as e:
                    pass

### Process Prompts lazily in a sequence

Notes:
1. Inspect http status from `call_llm`, advance to next prompt only on `HTTP 200`
2. Errors from `call_llm` (e.g `HTTP 429`) are handled via `throw(RetrySignal)`
3. `throw(RetrySignal)` yields the same prompt so that it can be retried on the LLM

In [75]:
def prompt_processor():
    prompt_generator = load_prompts()
    try:
        prompt = next(prompt_generator)
        while True:
            http_status = call_llm(prompt)
            if http_status != 200:
                prompt = prompt_generator.throw(RetrySignal)
                continue
            print(f'Successfully processed: {prompt}')
            prompt = next(prompt_generator)
    except StopIteration:
        pass

In [77]:
prompt_processor()

LLM call failed on prompt: What is Linear Regression ? - Please retry
LLM call failed on prompt: What is Linear Regression ? - Please retry
Successfully processed: What is Linear Regression ?

Successfully processed: Distinguish between Precision and Recall

Successfully processed: What is the formula for the soft-max function ?

LLM call failed on prompt: What is Simulated Annealing ? - Please retry
Successfully processed: What is Simulated Annealing ?

LLM call failed on prompt: What is the difference between a pmf and a pdf ? - Please retry
Successfully processed: What is the difference between a pmf and a pdf ?
